# Visualizacao do asset CHELSA PET (1991-2020) no Google Earth Engine

**Fluxo:**
1. Autentica e inicializa a API do GEE
2. Carrega o asset `CHELSA_pet_1991-2020` (imagem com 12 bandas, uma por mes)
3. Calcula uma escala de cores comum aos 12 meses
4. Renderiza os 12 mapas mensais lado a lado

## 1. Configuracao

In [ ]:
import ee
import geemap.colormaps as cm
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import requests
import folium
from io import BytesIO

PROJECT  = 'fcoliveira'  # projeto GEE dono do asset (ajuste se for outro)
ASSET_ID = 'projects/fcoliveira/assets/CHELSA_pet_1991-2020'

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print('GEE inicializado OK')

## 2. Carregar o asset

In [ ]:
img = ee.Image(ASSET_ID)

BANDS = img.bandNames().getInfo()
MESES = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

print(f'{len(BANDS)} bandas encontradas: {BANDS}')
assert len(BANDS) == 12, 'Esperava 12 bandas (uma por mes)'

# Bounding box da America do Sul (mesmo usado no download_chelsa_sa.py)
REGION = ee.Geometry.Rectangle([-82, -56, -34, 13])

## 3. Escala de cores comum aos 12 meses

Calcula percentis 2-98 de cada banda e usa o min/max global entre elas, para que as 12 imagens fiquem comparaveis na mesma paleta.

In [ ]:
stats = img.reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=REGION,
    scale=20000,
    bestEffort=True,
    maxPixels=1e9,
).getInfo()

p2_vals  = [stats[f'{b}_p2']  for b in BANDS]
p98_vals = [stats[f'{b}_p98'] for b in BANDS]
VIS_MIN, VIS_MAX = min(p2_vals), max(p98_vals)
PALETTE = list(cm.palettes.YlOrRd.default)

print(f'Escala de visualizacao: min={VIS_MIN:.1f}  max={VIS_MAX:.1f}')

## 4. Visualizar os 12 mapas mensais

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 18))

for band, mes, ax in zip(BANDS, MESES, axes.flat):
    thumb_url = img.select(band).getThumbURL({
        'region': REGION,
        'dimensions': 512,
        'min': VIS_MIN,
        'max': VIS_MAX,
        'palette': PALETTE,
    })
    resp = requests.get(thumb_url)
    thumb_img = mpimg.imread(BytesIO(resp.content), format='png')
    ax.imshow(thumb_img)
    ax.set_title(f'PET — {mes}', fontsize=12)
    ax.axis('off')
    print(f'  {mes} OK')

sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=VIS_MIN, vmax=VIS_MAX))
cbar = fig.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.03, pad=0.02)
cbar.set_label('PET (mm/mes)')

fig.suptitle('CHELSA PET — Normais Climatologicas Mensais (1991-2020)', fontsize=16)
plt.savefig('pet_12_meses.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Mapa interativo (folium) — navegar como no GEE

Em vez de miniaturas estaticas, cria um mapa `folium` de verdade: da para arrastar, dar zoom e ligar/desligar cada mes pelo painel de camadas, como no Code Editor do GEE.

In [ ]:
def add_ee_layer(self, ee_image, vis_params, name, show=True, opacity=1.0):
    map_id_dict = ee.Image(ee_image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name=name,
        overlay=True,
        control=True,
        show=show,
        opacity=opacity,
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

In [ ]:
VIS_PARAMS = {'min': VIS_MIN, 'max': VIS_MAX, 'palette': PALETTE}

m = folium.Map(location=[-15, -58], zoom_start=3, tiles='CartoDB positron')

for i, (band, mes) in enumerate(zip(BANDS, MESES)):
    m.add_ee_layer(img.select(band), VIS_PARAMS, name=f'PET — {mes}', show=(i == 0))

folium.LayerControl(collapsed=False).add_to(m)
m

## 6. Precipitacao (pr) e Temperatura do ar (tas)

> **Nota:** os dois asset IDs enviados eram identicos (`CHELSA_pr_1991-2020` duas vezes). Assumi que o de temperatura e `CHELSA_tas_1991-2020`, seguindo o padrao `pr`/`pet`/`tas` usado no resto do projeto (`CLAUDE.md`). Ajuste `ASSET_TAS` abaixo se o nome real for outro.

As funcoes abaixo generalizam o que foi feito para o PET (secoes 2-5), para nao repetir o mesmo bloco de codigo para cada variavel.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


def carregar_variavel(asset_id):
    img = ee.Image(asset_id)
    bands = img.bandNames().getInfo()
    print(f'{len(bands)} bandas encontradas em {asset_id}: {bands}')
    assert len(bands) == 12, 'Esperava 12 bandas (uma por mes)'
    return img, bands


def escala_de_cores(img, bands, region=REGION, scale=20000):
    stats = img.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
    ).getInfo()
    p2_vals  = [stats[f'{b}_p2']  for b in bands]
    p98_vals = [stats[f'{b}_p98'] for b in bands]
    return min(p2_vals), max(p98_vals)


def grade_12_mapas(img, bands, vis_min, vis_max, palette, titulo, unidade, out_path, region=REGION):
    fig, axes = plt.subplots(4, 3, figsize=(15, 18))
    for band, mes, ax in zip(bands, MESES, axes.flat):
        thumb_url = img.select(band).getThumbURL({
            'region': region,
            'dimensions': 512,
            'min': vis_min,
            'max': vis_max,
            'palette': palette,
        })
        resp = requests.get(thumb_url)
        thumb_img = mpimg.imread(BytesIO(resp.content), format='png')
        ax.imshow(thumb_img)
        ax.set_title(f'{titulo} — {mes}', fontsize=12)
        ax.axis('off')

    cmap = LinearSegmentedColormap.from_list('ee_palette', ['#' + h for h in palette])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vis_min, vmax=vis_max))
    cbar = fig.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.03, pad=0.02)
    cbar.set_label(unidade)

    fig.suptitle(f'CHELSA {titulo} — Normais Climatologicas Mensais (1991-2020)', fontsize=16)
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()


def mapa_interativo(img, bands, vis_min, vis_max, palette, titulo, center=(-15, -58)):
    vis_params = {'min': vis_min, 'max': vis_max, 'palette': palette}
    m = folium.Map(location=list(center), zoom_start=3, tiles='CartoDB positron')
    for i, (band, mes) in enumerate(zip(bands, MESES)):
        m.add_ee_layer(img.select(band), vis_params, name=f'{titulo} — {mes}', show=(i == 0))
    folium.LayerControl(collapsed=False).add_to(m)
    return m

### 6.1 Precipitacao (pr)

In [ ]:
ASSET_PR = 'projects/fcoliveira/assets/CHELSA_pr_1991-2020'
PALETTE_PR = list(cm.palettes.Blues.default)

img_pr, bands_pr = carregar_variavel(ASSET_PR)
vis_min_pr, vis_max_pr = escala_de_cores(img_pr, bands_pr)
print(f'Precipitacao — escala: min={vis_min_pr:.1f}  max={vis_max_pr:.1f}')

In [ ]:
grade_12_mapas(
    img_pr, bands_pr, vis_min_pr, vis_max_pr, PALETTE_PR,
    titulo='Precipitacao', unidade='pr (mm/mes)', out_path='pr_12_meses.png',
)

In [ ]:
mapa_interativo(img_pr, bands_pr, vis_min_pr, vis_max_pr, PALETTE_PR, titulo='Precipitacao')

### 6.2 Temperatura do ar (tas)

In [ ]:
ASSET_TAS = 'projects/fcoliveira/assets/CHELSA_tas_1991-2020'  # confirme se o nome do asset e esse mesmo
PALETTE_TAS = list(cm.palettes.RdYlBu.default)[::-1]  # azul (frio) -> vermelho (quente)

img_tas, bands_tas = carregar_variavel(ASSET_TAS)
vis_min_tas, vis_max_tas = escala_de_cores(img_tas, bands_tas)
print(f'Temperatura — escala: min={vis_min_tas:.1f}  max={vis_max_tas:.1f}')
print('Confira se esses valores fazem sentido em Celsius: o CHELSA guarda tas com fator de\n'
      'escala/offset proprio, e o pipeline de download/normais deste projeto nao aplica essa\n'
      'conversao — os valores acima podem estar na unidade bruta do arquivo original.')

In [ ]:
grade_12_mapas(
    img_tas, bands_tas, vis_min_tas, vis_max_tas, PALETTE_TAS,
    titulo='Temperatura do ar', unidade='tas (unidade original CHELSA)', out_path='tas_12_meses.png',
)

In [ ]:
mapa_interativo(img_tas, bands_tas, vis_min_tas, vis_max_tas, PALETTE_TAS, titulo='Temperatura do ar')